In [128]:
import os
import json
from datetime import datetime
from pprint import pprint

In [129]:
LOG_PATH="/Users/jarek/.local/share/opencode/log/dev.log"
PROMPTS_FILE_NAME=os.path.splitext(os.path.basename(LOG_PATH))[0]
START_TIME=datetime(2025, 11, 12, 0, 50)
OUTPUT_PATH=f"./logs/{PROMPTS_FILE_NAME}-prompts-from-{str(START_TIME).replace('-', '').replace(' ', '-').replace(':', '')}.log"
print(OUTPUT_PATH)

./logs/dev-prompts-from-20251112-005000.log


In [130]:
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

In [131]:
with open(LOG_PATH, mode='r') as f:
    log_lines = f.readlines()

In [132]:
len(log_lines)

2126

In [133]:
print(log_lines[:3])

['INFO  2025-11-12T00:02:44 +21ms service=default version=local args=[] opencode\n', 'INFO  2025-11-12T00:02:44 +1ms service=project directory=/Users/jarek/src/projects/opencode fromDirectory\n', 'INFO  2025-11-12T00:02:44 +20ms service=config path=/Users/jarek/.config/opencode/config.json loading\n']


In [171]:
LLM_REQUEST_MARKER="LLM request"
LLM_RESPONSE_MARKER="LLM response"
request_start_marker = 'providerID=' # system=[, messages=[, tools=[
tools_start_marker = "tools=["

response_start_marker = 'parts=['
PREVIOUS_PROMPT_COPY_MARKER="(.... PREVIOUS PROMPT ....)\n\n"

LINE_LENGTH=120

texts = []
for l in log_lines:
    line_time=l[6:25]
    try:
        dt = datetime.fromisoformat(line_time)
    except:
        continue

    l = l.strip()
    is_request = l.endswith(LLM_REQUEST_MARKER)
    is_response = l.endswith(LLM_RESPONSE_MARKER)
    #print(dt)
    if dt >= START_TIME and (is_request or is_response):
        if is_request:
            print("request...")
            p1 = l.index(request_start_marker) + len(request_start_marker)
            tt = 'REQUEST'
            entry: str = l[p1:-len(LLM_REQUEST_MARKER)].replace('\\n', '\n').split('\n')

            #chunks = len(entry)
            #entry = [ entry[i:i+LINE_LENGTH] for i in range(0, chunks, LINE_LENGTH) ]
            
            entry = [e + "\n" for e in entry]
            #print(len(entry))
        elif is_response:
            print("response...")
            p1 = l.index(response_start_marker) + len(response_start_marker)
            tt = 'RESPONSE'
            entry = l[p1-1:-len(LLM_RESPONSE_MARKER)]                        
            
            entry = json.loads(entry)            
        texts.append([dt, tt, entry])

request...
response...


In [172]:
len(texts)

2

In [173]:
#texts[0]

In [174]:
DIVIDER_FORMAT="\n=====================\n=== END OF {type} ===\n=====================\n"
with open(OUTPUT_PATH, mode='w') as f:
    line = 0
    for dt, tt, entry in texts:
        line += 1
        print(f"Storing #{line}")
        f.write(f"{dt}, {tt}\n")
        if tt == "REQUEST":
            f.writelines(entry)
        else:
            f.write(json.dumps(entry, indent=2))
        f.write(DIVIDER.replace('{type}', tt))

Storing #1
Storing #2
